# PDRB Triwulanan — Data Cleaning & Consolidation
**[Seri 2010] PDRB Atas Dasar Harga Berlaku (ADHB) & Atas Dasar Harga Konstan (ADHK)  
Menurut Lapangan Usaha, Seluruh Provinsi Indonesia**

This notebook:
1. Parses the raw BPS CSV files (one per release year, 2020–2026) for **both** price bases
2. Tags each row with `price_basis` → `'ADHB'` (current prices) or `'ADHK'` (constant 2010 prices)
3. Extracts province × sector × quarter values
4. Concatenates into a single long-format panel covering both series
5. Exports clean datasets: long format and a wide pivot

**Values are in Milyar Rupiah (IDR billion).**  
ADHB = Atas Dasar Harga Berlaku (current prices) | ADHK = Atas Dasar Harga Konstan 2010 (constant prices)


## 0 — Imports & file paths

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# ── Set DATA_DIR to the folder containing all BPS CSV files ─────────────────
DATA_DIR = Path(r'C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\00_base_data')

# Separate file lists by price basis (detected from filename keyword)
ALL_FILES  = sorted(DATA_DIR.glob('*PDRB Triwulanan*.csv'))
FILES_ADHB = [f for f in ALL_FILES if 'Harga Berlaku'  in f.name]
FILES_ADHK = [f for f in ALL_FILES if 'Harga Konstan'  in f.name]

print(f'ADHB files ({len(FILES_ADHB)}):')
for f in FILES_ADHB: print(' ', f.name)
print(f'\nADHK files ({len(FILES_ADHK)}):')
for f in FILES_ADHK: print(' ', f.name)


KeyboardInterrupt: 

## 1 — Raw file structure

Each BPS CSV has **5 header rows** (0-indexed) followed by province data:

| Row | Content |
|-----|--------------------------------------------------------|
| 0   | `'Provinsi'` in col 0, rest NaN |
| 1   | Full dataset title in col 1 |
| 2   | Sector names (one every 5 cols starting at col 1) |
| 3   | Year repeated per sector group |
| 4   | Quarter labels: Triwulan I/II/III/IV + Tahunan (5 per sector) |
| 5+  | Province data rows |

**91 columns total:** col 0 = province name, then 18 sectors × 5 time slots = 90 value columns.  
The structure is **identical** for ADHB and ADHK files.


In [ ]:
# Sanity check on one ADHB and one ADHK file
for label, flist in [('ADHB', FILES_ADHB), ('ADHK', FILES_ADHK)]:
    sample = pd.read_csv(flist[1], header=None, dtype=str)  # use the 2021 release
    print(f'── {label} ──')
    print('  Shape    :', sample.shape)
    print('  Row 3 years (unique):', sample.iloc[3, 1:].dropna().unique().tolist())
    print('  Row 4 quarters (cols 1-6):', sample.iloc[4, 1:7].tolist())
    print()


── ADHB ──
  Shape    : (43, 91)
  Row 3 years (unique): ['2021']
  Row 4 quarters (cols 1-6): ['Triwulan I', 'Triwulan II', 'Triwulan III', 'Triwulan IV', 'Tahunan', 'Triwulan I']

── ADHK ──
  Shape    : (43, 91)
  Row 3 years (unique): ['2021']
  Row 4 quarters (cols 1-6): ['Triwulan I', 'Triwulan II', 'Triwulan III', 'Triwulan IV', 'Tahunan', 'Triwulan I']



## 2 — Constants

In [ ]:
SECTOR_CODES = [
    'A',     # Pertanian, Kehutanan dan Perikanan
    'B',     # Pertambangan dan Penggalian
    'C',     # Industri Pengolahan
    'D',     # Pengadaan Listrik dan Gas
    'E',     # Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang
    'F',     # Konstruksi
    'G',     # Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor
    'H',     # Transportasi dan Pergudangan
    'I',     # Penyediaan Akomodasi dan Makan Minum
    'J',     # Informasi dan Komunikasi
    'K',     # Jasa Keuangan dan Asuransi
    'L',     # Real Estate
    'MN',    # Jasa Perusahaan
    'O',     # Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib
    'P',     # Jasa Pendidikan
    'Q',     # Jasa Kesehatan dan Kegiatan Sosial
    'RSTU',  # Jasa Lainnya
    'PDRB',  # Produk Domestik Regional Bruto (total)
]

SECTOR_NAMES = [
    'Pertanian, Kehutanan dan Perikanan',
    'Pertambangan dan Penggalian',
    'Industri Pengolahan',
    'Pengadaan Listrik dan Gas',
    'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang',
    'Konstruksi',
    'Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor',
    'Transportasi dan Pergudangan',
    'Penyediaan Akomodasi dan Makan Minum',
    'Informasi dan Komunikasi',
    'Jasa Keuangan dan Asuransi',
    'Real Estate',
    'Jasa Perusahaan',
    'Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib',
    'Jasa Pendidikan',
    'Jasa Kesehatan dan Kegiatan Sosial',
    'Jasa Lainnya',
    'Produk Domestik Regional Bruto',
]

QUARTER_MAP = {
    'Triwulan I':   'Q1',
    'Triwulan II':  'Q2',
    'Triwulan III': 'Q3',
    'Triwulan IV':  'Q4',
}

N_SECTORS = 18
N_PERIODS = 5   # Q1, Q2, Q3, Q4, Tahunan per sector per year

print(f'Sectors: {N_SECTORS} | Periods per sector: {N_PERIODS}')
print(f'Expected value columns: {N_SECTORS * N_PERIODS} (+1 province col = 91 total)')


Sectors: 18 | Periods per sector: 5
Expected value columns: 90 (+1 province col = 91 total)


## 3 — Parser function

In [ ]:
def parse_pdrb_file(filepath: Path, price_basis: str) -> pd.DataFrame:
    """
    Parse one BPS PDRB CSV and return a long-format DataFrame with columns:
        price_basis | provinsi | year | quarter | period | sector_code | sector_name | value_billion_idr

    Parameters
    ----------
    filepath    : path to the BPS CSV file
    price_basis : 'ADHB' (current prices) or 'ADHK' (constant 2010 prices)

    Notes
    -----
    - 'Tahunan' (annual total) columns are dropped; only Q1–Q4 are retained.
    - Dash ('-') and blank cells become NaN.
    - Files with no province rows return an empty DataFrame.
    """
    raw = pd.read_csv(filepath, header=None, dtype=str)

    # --- 1. Build (year, quarter) metadata for each of the 90 value columns ---
    year_row    = raw.iloc[3, 1:].fillna('').astype(str)
    quarter_row = raw.iloc[4, 1:].fillna('').astype(str)

    col_meta = []
    current_year = None
    for yr, qt in zip(year_row, quarter_row):
        yr = yr.strip()
        qt = qt.strip()
        if re.match(r'^\d{4}$', yr):
            current_year = int(yr)
        col_meta.append((current_year, qt))

    # --- 2. Extract province data rows (start at row index 5) ----------------
    data_rows = raw.iloc[5:].copy()
    data_rows.columns = ['provinsi'] + list(range(len(raw.columns) - 1))
    data_rows = data_rows.dropna(subset=['provinsi']).reset_index(drop=True)

    if data_rows.empty:
        print(f'  [SKIP] {filepath.name} — no province data rows')
        return pd.DataFrame(columns=[
            'price_basis', 'provinsi', 'year', 'quarter', 'period',
            'sector_code', 'sector_name', 'value_billion_idr'
        ])

    # --- 3. Melt wide -> long -------------------------------------------------
    records = []
    for _, row in data_rows.iterrows():
        prov = str(row['provinsi']).strip()
        for col_idx, (year, qt_label) in enumerate(col_meta):
            if qt_label not in QUARTER_MAP:
                continue  # skip 'Tahunan' and blank labels
            sector_idx = col_idx // N_PERIODS
            if sector_idx >= len(SECTOR_CODES):
                continue
            raw_val = str(row[col_idx]).strip()
            try:
                value = (
                    np.nan
                    if raw_val in ('-', '', 'nan', 'None')
                    else float(raw_val.replace(',', ''))
                )
            except (ValueError, TypeError):
                value = np.nan
            records.append({
                'price_basis':       price_basis,
                'provinsi':          prov,
                'year':              year,
                'quarter':           QUARTER_MAP[qt_label],
                'sector_code':       SECTOR_CODES[sector_idx],
                'sector_name':       SECTOR_NAMES[sector_idx],
                'value_billion_idr': value,
            })

    df = pd.DataFrame(records)
    df['period'] = df['year'].astype(str) + df['quarter']
    return df

print('Parser ready.')


Parser ready.


## 4 — Parse all files (ADHB + ADHK)

In [ ]:
all_frames = []

for price_basis, file_list in [('ADHB', FILES_ADHB), ('ADHK', FILES_ADHK)]:
    print(f'── Parsing {price_basis} ──')
    for f in file_list:
        df_f = parse_pdrb_file(f, price_basis=price_basis)
        if df_f.empty:
            continue
        years  = sorted(df_f['year'].dropna().unique())
        n_prov = df_f['provinsi'].nunique()
        print(f'  {f.name[-10:]} -> {len(df_f):6,} rows | years={years} | provinces={n_prov}')
        all_frames.append(df_f)
    print()

raw_combined = pd.concat(all_frames, ignore_index=True)
print(f'Combined total: {len(raw_combined):,} rows')
print('Price bases:', raw_combined['price_basis'].value_counts().to_dict())


── Parsing ADHB ──
  , 2020.csv ->  2,736 rows | years=[np.int64(2020)] | provinces=38
  , 2021.csv ->  2,736 rows | years=[np.int64(2021)] | provinces=38
  , 2022.csv ->  2,736 rows | years=[np.int64(2022)] | provinces=38
  , 2023.csv ->  2,736 rows | years=[np.int64(2023)] | provinces=38
  , 2024.csv ->  2,736 rows | years=[np.int64(2024)] | provinces=38
  , 2025.csv ->  2,736 rows | years=[np.int64(2025)] | provinces=38
  , 2026.csv ->  2,736 rows | years=[np.int64(2026)] | provinces=38

── Parsing ADHK ──
  , 2020.csv ->  2,736 rows | years=[np.int64(2020)] | provinces=38
  , 2021.csv ->  2,736 rows | years=[np.int64(2021)] | provinces=38
  , 2022.csv ->  2,736 rows | years=[np.int64(2022)] | provinces=38
  , 2023.csv ->  2,736 rows | years=[np.int64(2023)] | provinces=38
  , 2024.csv ->  2,736 rows | years=[np.int64(2024)] | provinces=38
  , 2025.csv ->  2,736 rows | years=[np.int64(2025)] | provinces=38
  , 2026.csv ->  2,736 rows | years=[np.int64(2026)] | provinces=38

Combined

## 5 — Filter & deduplicate

In [ ]:
QUARTER_ORDER = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}
raw_combined['period_sort'] = (
    raw_combined['year'].astype(float) * 10
    + raw_combined['quarter'].map(QUARTER_ORDER)
)

# Filter to 2020Q1 – 2026Q1
# ADHK files run to 2025; ADHB files run to 2026Q1 — both are retained as-is.
TARGET_START = 2020 * 10 + 1   # 2020Q1
TARGET_END   = 2026 * 10 + 1   # 2026Q1

filtered = raw_combined[
    (raw_combined['period_sort'] >= TARGET_START) &
    (raw_combined['period_sort'] <= TARGET_END)
].copy()

print(f'Rows after date filter: {len(filtered):,}')
print('\nPeriods by price basis:')
print(
    filtered.groupby('price_basis')['period']
    .agg(lambda s: sorted(s.unique()))
    .to_string()
)


Rows after date filter: 34,200

Periods by price basis:
price_basis
ADHB    [2020Q1, 2020Q2, 2020Q3, 2020Q4, 2021Q1, 2021Q...
ADHK    [2020Q1, 2020Q2, 2020Q3, 2020Q4, 2021Q1, 2021Q...


In [ ]:
# Deduplicate within each price_basis: keep non-NaN over NaN
KEY_COLS = ['price_basis', 'provinsi', 'year', 'quarter', 'sector_code']

filtered_sorted = filtered.sort_values(
    by=KEY_COLS + ['value_billion_idr'],
    na_position='first'
)
deduped = (
    filtered_sorted
    .drop_duplicates(subset=KEY_COLS, keep='last')
    .reset_index(drop=True)
)

print(f'Rows after dedup : {len(deduped):,}')
print(f'Provinces        : {deduped["provinsi"].nunique()}')
print(f'Sectors          : {deduped["sector_code"].nunique()}')
print('\nRows per price_basis:')
print(deduped['price_basis'].value_counts().to_string())


Rows after dedup : 34,200
Provinces        : 38
Sectors          : 18

Rows per price_basis:
price_basis
ADHB    17100
ADHK    17100


## 6 — Data quality checks

In [ ]:
# Missing value summary by price basis
for pb in ['ADHB', 'ADHK']:
    sub   = deduped[deduped['price_basis'] == pb]
    total = len(sub)
    null_n = sub['value_billion_idr'].isna().sum()
    print(f'{pb}: {null_n:,} / {total:,} missing  ({null_n/total*100:.1f}%)')


ADHB: 1,152 / 17,100 missing  (6.7%)
ADHK: 1,152 / 17,100 missing  (6.7%)


In [ ]:
# Coverage matrix: PDRB total per province-period, by price basis
for pb in ['ADHB', 'ADHK']:
    sub = deduped[(deduped['price_basis'] == pb) & (deduped['sector_code'] == 'PDRB')]
    coverage = sub.pivot_table(
        index='provinsi',
        columns='period',
        values='value_billion_idr',
        aggfunc=lambda x: 'OK' if x.notna().any() else 'MISSING'
    )
    print(f'\n── Coverage matrix ({pb}) ──')
    display(coverage)



── Coverage matrix (ADHB) ──


period,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4,2022Q1,2022Q2,...,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1
provinsi,,,,,,,,,,,,,,,,,,,,,
Aceh,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Bali,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Banten,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Bengkulu,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
DI Yogyakarta,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
DKI Jakarta,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Gorontalo,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Jambi,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Jawa Barat,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK



── Coverage matrix (ADHK) ──


period,2020Q1,2020Q2,2020Q3,2020Q4,2021Q1,2021Q2,2021Q3,2021Q4,2022Q1,2022Q2,...,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1
provinsi,,,,,,,,,,,,,,,,,,,,,
Aceh,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Bali,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Banten,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Bengkulu,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
DI Yogyakarta,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
DKI Jakarta,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Gorontalo,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Jambi,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Jawa Barat,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,...,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK


In [ ]:
# Verify: sector sum ≈ PDRB total, for each price basis
for pb in ['ADHB', 'ADHK']:
    sub = deduped[deduped['price_basis'] == pb]
    totals = (
        sub[sub['sector_code'] == 'PDRB']
        [['provinsi', 'period', 'value_billion_idr']]
        .rename(columns={'value_billion_idr': 'pdrb_total'})
    )
    sector_sums = (
        sub[sub['sector_code'] != 'PDRB']
        .groupby(['provinsi', 'period'])['value_billion_idr']
        .sum(min_count=1)
        .reset_index()
        .rename(columns={'value_billion_idr': 'sector_sum'})
    )
    check = totals.merge(sector_sums, on=['provinsi', 'period'])
    check['diff_pct'] = (check['pdrb_total'] - check['sector_sum']).abs() / check['pdrb_total'] * 100
    large_diff = check[check['diff_pct'] > 1.0].sort_values('diff_pct', ascending=False)
    print(f'{pb}: province-period pairs with |PDRB - sum of sectors| > 1%: {len(large_diff)}')
    if len(large_diff) > 0:
        print(large_diff.head(5).to_string(index=False))
    print()


ADHB: province-period pairs with |PDRB - sum of sectors| > 1%: 0

ADHK: province-period pairs with |PDRB - sum of sectors| > 1%: 0



## 7 — Finalise & export

In [ ]:
# Final dataset: tidy column order and dtypes
final = (
    deduped
    .sort_values(['price_basis', 'provinsi', 'period_sort', 'sector_code'])
    [['price_basis', 'provinsi', 'year', 'quarter', 'period',
      'sector_code', 'sector_name', 'value_billion_idr']]
    .reset_index(drop=True)
)
final['year'] = final['year'].astype('Int16')
final['value_billion_idr'] = pd.to_numeric(final['value_billion_idr'], errors='coerce')

print('Final dataset shape:', final.shape)
print('\nDtypes:')
print(final.dtypes)
print('\nSample (first 5 rows):')
final.head()


Final dataset shape: (34200, 8)

Dtypes:
price_basis           object
provinsi              object
year                   Int16
quarter               object
period                object
sector_code           object
sector_name           object
value_billion_idr    float64
dtype: object

Sample (first 5 rows):


,price_basis,provinsi,year,quarter,period,sector_code,sector_name,value_billion_idr
0,ADHB,Aceh,2020,Q1,2020Q1,A,"Pertanian, Kehutanan dan Perikanan",13233.90
1,ADHB,Aceh,2020,Q1,2020Q1,B,Pertambangan dan Penggalian,1723.62
2,ADHB,Aceh,2020,Q1,2020Q1,C,Industri Pengolahan,1727.17
3,ADHB,Aceh,2020,Q1,2020Q1,D,Pengadaan Listrik dan Gas,56.51
4,ADHB,Aceh,2020,Q1,2020Q1,E,"Pengadaan Air, Pengelolaan Sampah, Limbah dan ...",19.58


In [ ]:
OUT_DIR = Path(r'C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data')

# Long format (both price bases in one file)
out_long = OUT_DIR / '02_01_pdrb_sectoral_long.csv'
final.to_csv(out_long, index=False)
print(f'Long format saved -> {out_long}  ({out_long.stat().st_size/1024:.0f} KB, {len(final):,} rows)')

# Optional: separate files per price basis
for pb in ['ADHB', 'ADHK']:
    sub = final[final['price_basis'] == pb]
    out_pb = OUT_DIR / f'02_01_pdrb_sectoral_long_{pb.lower()}.csv'
    sub.to_csv(out_pb, index=False)
    print(f'{pb} saved  -> {out_pb}  ({out_pb.stat().st_size/1024:.0f} KB, {len(sub):,} rows)')


Long format saved -> C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data\02_01_pdrb_sectoral_long.csv  (2482 KB, 34,200 rows)
ADHB saved  -> C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data\02_01_pdrb_sectoral_long_adhb.csv  (1242 KB, 17,100 rows)
ADHK saved  -> C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data\02_01_pdrb_sectoral_long_adhk.csv  (1239 KB, 17,100 rows)


## 8 — Summary statistics

In [ ]:
# National aggregate PDRB by quarter, for both price bases
national = (
    final[final['sector_code'] == 'PDRB']
    .groupby(['price_basis', 'period'])['value_billion_idr']
    .sum()
    .unstack('price_basis')
    .rename(columns={'ADHB': 'ADHB_billion_idr', 'ADHK': 'ADHK_billion_idr'})
)
print('National aggregate PDRB (Milyar Rupiah):')
print(national.to_string())


National aggregate PDRB (Milyar Rupiah):
price_basis  ADHB_billion_idr  ADHK_billion_idr
period                                         
2020Q1             4022590.05        2757052.24
2020Q2             3755466.59        2601918.93
2020Q3             3972121.66        2734033.86
2020Q4             4014088.22        2744535.84
2021Q1             4054496.32        2736880.97
2021Q2             4184548.04        2789168.75
2021Q3             4283200.58        2830838.36
2021Q4             4429564.31        2882499.84
2022Q1             4490078.31        2869532.45
2022Q2             4756056.47        2945500.72
2022Q3             4896944.61        2994200.19
2022Q4             5001749.59        3030941.83
2023Q1             4986384.15        3014235.52
2023Q2             5100406.99        3099367.21
2023Q3             5176091.88        3142335.79
2023Q4             5270043.16        3183772.40
2024Q1             5284404.27        3168390.82
2024Q2             5498240.75        3256904.92

In [ ]:
# Implied deflator: ADHB / ADHK (rough GDP deflator proxy, index 2010 = 100)
national['deflator_index'] = national['ADHB_billion_idr'] / national['ADHK_billion_idr'] * 100
print('Implied GDP deflator (ADHB / ADHK × 100):')
print(national['deflator_index'].to_string())


Implied GDP deflator (ADHB / ADHK × 100):
period
2020Q1    145.901844
2020Q2    144.334497
2020Q3    145.284289
2020Q4    146.257453
2021Q1    148.142954
2021Q2    150.028500
2021Q3    151.305021
2021Q4    153.670930
2022Q1    156.474213
2022Q2    161.468522
2022Q3    163.547669
2022Q4    165.022949
2023Q1    165.427821
2023Q2    164.562849
2023Q3    164.721157
2023Q4    165.528263
2024Q1    166.785115
2024Q2    168.817969
2024Q3    168.934725
2024Q4    169.526519
2025Q1    170.998244
2025Q2    171.868271
2025Q3    172.297466
2025Q4    173.151430
2026Q1    175.776696


In [ ]:
# Sector shares in the latest available period — shown for ADHB
latest_adhb = sorted(final[final['price_basis'] == 'ADHB']['period'].unique())[-1]
print(f'Latest ADHB period: {latest_adhb}\n')

sector_share = (
    final[
        (final['price_basis']   == 'ADHB') &
        (final['period']        == latest_adhb) &
        (final['sector_code']   != 'PDRB')
    ]
    .groupby(['sector_code', 'sector_name'])['value_billion_idr']
    .sum()
    .reset_index()
)
total_val = sector_share['value_billion_idr'].sum()
sector_share['share_pct'] = sector_share['value_billion_idr'] / total_val * 100
sector_share = sector_share.sort_values('value_billion_idr', ascending=False)

print(f'Sector shares (ADHB) — {latest_adhb} (all provinces):')
print(sector_share[['sector_code', 'sector_name', 'value_billion_idr', 'share_pct']].to_string(index=False))


Latest ADHB period: 2026Q1

Sector shares (ADHB) — 2026Q1 (all provinces):
sector_code                                                    sector_name  value_billion_idr  share_pct
          C                                            Industri Pengolahan         1446465.96  23.451306
          G  Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor          910185.31  14.756679
          A                             Pertanian, Kehutanan dan Perikanan          800231.34  12.974014
          F                                                     Konstruksi          608766.74   9.869831
          B                                    Pertambangan dan Penggalian          403305.43   6.538722
          H                                   Transportasi dan Pergudangan          320653.16   5.198695
          J                                       Informasi dan Komunikasi          285118.06   4.622570
          K                                     Jasa Keuangan dan Asuransi          2

---
## Output files

| File | Description |
|------|-------------|
| `02_01_pdrb_sectoral_long.csv` | Long format: one row per **price_basis** × province × quarter × sector |
| `02_01_pdrb_sectoral_long_adhb.csv` | Long format, ADHB only (current prices) |
| `02_01_pdrb_sectoral_long_adhk.csv` | Long format, ADHK only (constant 2010 prices) |

**Unit:** Milyar Rupiah (IDR billion).  
**Coverage:** 38 provinces × 18 sectors × quarters from 2021Q1.  
**ADHB** runs to 2026Q1; **ADHK** runs to 2025Q4 (no 2026 file released yet).
